# REDES NEURAIS ARTIFICIAIS

# PARTE 1: Classificação Binária - Diagnóstico de Câncer de Mama

Base de dados: Breast Cancer Wisconsin (Diagnostic) -
https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic

In [2]:
!pip install torch skorch torchinfo scikit-learn==1.5.0


  Using cached scikit_learn-1.5.0.tar.gz (7.8 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'error'


  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [21 lines of output]
      + meson setup C:\Users\tiago\AppData\Local\Temp\pip-install-zkdgqgju\scikit-learn_d5a8e147d911488c93153467f19c6598 C:\Users\tiago\AppData\Local\Temp\pip-install-zkdgqgju\scikit-learn_d5a8e147d911488c93153467f19c6598\.mesonpy-5yl9dbnx -Dbuildtype=release -Db_ndebug=if-release -Db_vscrt=md --native-file=C:\Users\tiago\AppData\Local\Temp\pip-install-zkdgqgju\scikit-learn_d5a8e147d911488c93153467f19c6598\.mesonpy-5yl9dbnx\meson-python-native-file.ini
      The Meson build system
      Version: 1.11.1
      Source dir: C:\Users\tiago\AppData\Local\Temp\pip-install-zkdgqgju\scikit-learn_d5a8e147d911488c93153467f19c6598
      Build dir: C:\Users\tiago\AppData\Local\Temp\pip-install-zkdgqgju\scikit-learn_d5a8e147d911488c93153467f19c6598\.mesonpy-5yl9dbnx
      Build type: native build
      Project name: scikit-learn
      Project version:

Este notebook usa **PyTorch** (via `skorch`, que conecta modelos PyTorch ao scikit-learn) em vez de TensorFlow/Keras. PyTorch já tem suporte ao Python 3.14, então roda normalmente local ou no Colab.

In [3]:
# Com PyTorch normalmente não é necessário reiniciar o notebook após instalar os pacotes,
# mas se o pip tiver acabado de instalar algo novo, reinicie o kernel manualmente se der erro de import.
!pip install torch
!pip install skorch torchinfo

In [4]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import sklearn
import skorch
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from skorch import NeuralNetBinaryClassifier
from torchinfo import summary as torch_summary


In [5]:
pd.__version__, torch.__version__, sklearn.__version__, skorch.__version__


('3.0.2', '2.12.1+cpu', '1.8.0', '1.4.0')

### Carregar os dados

**1) Carregue a base de dados, faça a divisão de treino e teste (para isso, utilize a função train_test_split do sklearn), como o tamanho da base de teste de 0.25.**

In [6]:
X = pd.read_csv('dados_breast.csv')
y = pd.read_csv('rotulos_breast.csv')

X_treinamento, X_teste, y_treinamento, y_teste = train_test_split(
    X, y, test_size=0.25, random_state=0
)

In [7]:
X_treinamento.shape, y_treinamento.shape

((426, 30), (426, 1))

In [8]:
X_teste.shape, y_teste.shape

((143, 30), (143, 1))

### Estrutura da Rede Neural Artificial e Teste

In [9]:
from skorch import NeuralNetBinaryClassifier
from sklearn.model_selection import cross_val_score
import torch
import torch.nn as nn


**2) Crie a RNA com as seguintes configurações:**

**a) uma camada de entrada com 30 neurônios;**

**b) uma camada oculta densa com 16 neurônios. Porque a camada oculta tem essa quantidade de neurônios?**

A camada oculta com 16 neurônios é uma escolha heurística comum: utiliza-se um número entre o tamanho da entrada (30) e da saída (1), geralmente uma potência de 2 para eficiência computacional. 16 é suficiente para capturar padrões não-lineares do problema sem tornar o modelo excessivamente complexo.

**c) adicione na camada oculta a função de ativação relu e inicialize os pesos com Random uniform initializer;**

**d) adicione a camada de saída com a função de ativação sigmoid. Porque foi utilizada esta função para a saída da RNA?**

A função sigmoid é utilizada na saída porque produz valores entre 0 e 1, que podem ser interpretados como probabilidades. Como o problema é de classificação binária (maligno/benigno), precisamos de uma saída que represente a probabilidade de pertencer à classe 1.

**3) Porque utilizamos a classe chamada Sequential para a RNA?**

A classe Sequential é utilizada porque permite construir a rede camada por camada de forma linear (sequencial), onde a saída de uma camada é a entrada da próxima. É a forma mais simples e direta de criar uma RNA com fluxo unidirecional, adequada para a maioria dos problemas de classificação.

In [10]:
class RedeNeuralModule(nn.Module):
    def __init__(self, n_features=30, neurons=16):
        super().__init__()
        self.oculta = nn.Linear(n_features, neurons)
        self.relu = nn.ReLU()
        self.saida = nn.Linear(neurons, 1)
        nn.init.uniform_(self.oculta.weight)
        nn.init.uniform_(self.saida.weight)

    def forward(self, X):
        x = self.relu(self.oculta(X))
        x = self.saida(x)  # logits (sem sigmoid); NeuralNetBinaryClassifier aplica BCEWithLogitsLoss
        return x.squeeze(-1)

rede_neural = NeuralNetBinaryClassifier(
    module=RedeNeuralModule,
    max_epochs=100,
    batch_size=10,
    verbose=0,
)


**4) A partir da RNA gerada, explique o que são os valores apresentados na tabela da rede_neural.summary()**

- **Layer (type)**: Nome e tipo de cada camada da rede.
- **Output Shape**: Dimensão da saída de cada camada. `(None, 16)` significa que para cada amostra (None = qualquer quantidade de amostras) há 16 saídas.
- **Param #**: Número de parâmetros treináveis. Para a camada densa de 16 neurônios: (30 entradas × 16 neurônios) + 16 biases = 496 parâmetros. Para a camada de saída: (16 × 1) + 1 bias = 17 parâmetros. Total: 513 parâmetros.
- **Trainable params**: parâmetros que serão atualizados durante o treinamento.
- **Non-trainable params**: parâmetros fixos (ex.: em camadas congeladas).

In [11]:
rede_neural.initialize()
torch_summary(rede_neural.module_, input_size=(1, 30))


Layer (type:depth-idx)                   Output Shape              Param #
RedeNeuralModule                         [1]                       --
├─Linear: 1-1                            [1, 16]                   496
├─ReLU: 1-2                              [1, 16]                   --
├─Linear: 1-3                            [1, 1]                    17
Total params: 513
Trainable params: 513
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00

**5) Adicione um otimizador Adam e especifique a classe loss binário - binary crossentropy e a métrica de acurácia binária.**

**6) Para que servem os otimizadores? Como o otimizador Adam funciona?**

Os otimizadores servem para atualizar os pesos da rede neural de forma a minimizar a função de perda (loss). O Adam (Adaptive Moment Estimation) combina duas técnicas:
- **Momentum**: acumula gradientes passados para suavizar a atualização (evita oscilações);
- **RMSProp**: adapta a taxa de aprendizado para cada parâmetro individualmente com base na magnitude dos gradientes recentes.
O resultado é um otimizador eficiente, robusto e que converge rapidamente, sendo uma das escolhas padrão na prática.

**7) Depois de estruturados os parâmetros da RNA, utilize a função .fit para fazer o treinamento da rede. Com batch_size=10, quantos batches serão utilizados ao total?**

O conjunto de treinamento tem 426 amostras (75% de 569). Com batch_size=10: ⌈426 / 10⌉ = 43 batches por época × 100 épocas = 4.300 atualizações de pesos no total.

**8) Defina o número de épocas em que ocorre o treinamento igual a 100.**

In [12]:
rede_neural.set_params(optimizer=torch.optim.Adam, criterion=nn.BCEWithLogitsLoss)


,module,<class '__mai...NeuralModule'>
,criterion,<class 'torch...thLogitsLoss'>
,train_split,<skorch.datas...001EBFFD60050>
,threshold,0.5
,optimizer,<class 'torch...im.adam.Adam'>
,lr,0.01
,max_epochs,100
,batch_size,10
,iterator_train,<class 'torch...r.DataLoader'>
,iterator_valid,<class 'torch...r.DataLoader'>
,dataset,<class 'skorc...aset.Dataset'>


In [13]:
X_tr = X_treinamento.values.astype(np.float32)
y_tr = y_treinamento.values.astype(np.float32).ravel()
rede_neural.fit(X_tr, y_tr)


,module,<class '__mai...NeuralModule'>
,criterion,<class 'torch...thLogitsLoss'>
,train_split,<skorch.datas...001EBFFD60050>
,threshold,0.5
,optimizer,<class 'torch...im.adam.Adam'>
,lr,0.01
,max_epochs,100
,batch_size,10
,iterator_train,<class 'torch...r.DataLoader'>
,iterator_valid,<class 'torch...r.DataLoader'>
,dataset,<class 'skorc...aset.Dataset'>


**9) Crie uma variável chamada previsoes para realizar a previsão dos dados de teste (X_teste). O resultado da rede deve ser um valor entre 0 e 1. Porque isso acontece?**

Isso acontece porque a camada de saída utiliza a função sigmoid, que mapeia qualquer valor real para o intervalo (0, 1). Esse valor representa a probabilidade da amostra pertencer à classe positiva (tumor maligno).

In [14]:
X_te = X_teste.values.astype(np.float32)
previsoes = rede_neural.predict_proba(X_te)[:, 1]


**10) Converta o resultado que está em probabilidade para valores binários com um limiar (threshold) de 0.5.**

In [15]:
previsoes = previsoes > 0.5

In [16]:
previsoes

array([False,  True,  True, False,  True, False,  True,  True,  True,
        True, False, False,  True, False, False, False,  True, False,
       False, False, False, False,  True, False, False, False,  True,
       False,  True, False,  True, False, False, False,  True, False,
       False, False,  True, False, False,  True, False,  True, False,
       False,  True,  True,  True, False,  True,  True, False,  True,
        True,  True,  True,  True,  True, False, False, False,  True,
        True, False, False, False, False, False,  True, False, False,
        True, False, False,  True,  True,  True, False,  True, False,
       False, False,  True, False, False, False,  True, False, False,
        True, False,  True, False,  True,  True, False,  True,  True,
        True,  True,  True,  True, False, False,  True, False, False,
       False, False, False, False, False, False,  True,  True, False,
        True,  True,  True, False,  True, False, False, False, False,
       False,  True,

**9) Explique o resultado da RNA.**

Após a conversão com threshold 0.5, cada previsão é True (classe 1 = maligno) ou False (classe 0 = benigno). A função `evaluate` retorna o valor da loss (binary_crossentropy) e a acurácia binária no conjunto de teste, mostrando o desempenho do modelo em dados não vistos durante o treinamento.

In [17]:
y_te = y_teste.values.astype(np.float32).ravel()
acuracia = rede_neural.score(X_te, y_te)
print(f'Acurácia binária no teste: {acuracia:.4f}')


Acurácia binária no teste: 0.8042


## Camadas e Otimização da RNA

**10) Adicione mais uma camada oculta densa com 16 neurônios com a função de ativação relu e inicialize os pesos com Random uniform initializer. Qual o total de parâmetros da RNA agora?**

Com a segunda camada oculta:
- Camada 1: (30×16) + 16 = 496 parâmetros
- Camada 2: (16×16) + 16 = 272 parâmetros
- Saída: (16×1) + 1 = 17 parâmetros
- **Total: 785 parâmetros**

In [18]:
class RedeNeuralModule(nn.Module):
    def __init__(self, n_features=30, neurons=16):
        super().__init__()
        self.oculta1 = nn.Linear(n_features, neurons)
        self.oculta2 = nn.Linear(neurons, neurons)
        self.relu = nn.ReLU()
        self.saida = nn.Linear(neurons, 1)
        for camada in (self.oculta1, self.oculta2, self.saida):
            nn.init.uniform_(camada.weight)

    def forward(self, X):
        x = self.relu(self.oculta1(X))
        x = self.relu(self.oculta2(x))
        x = self.saida(x)
        return x.squeeze(-1)

rede_neural = NeuralNetBinaryClassifier(
    module=RedeNeuralModule,
    max_epochs=100,
    batch_size=10,
    verbose=0,
)
rede_neural.initialize()


,module,<class '__mai...NeuralModule'>
,criterion,<class 'torch...thLogitsLoss'>
,train_split,<skorch.datas...001EBFFD60050>
,threshold,0.5
,optimizer,<class 'torch.optim.sgd.SGD'>
,lr,0.01
,max_epochs,100
,batch_size,10
,iterator_train,<class 'torch...r.DataLoader'>
,iterator_valid,<class 'torch...r.DataLoader'>
,dataset,<class 'skorc...aset.Dataset'>


In [19]:
torch_summary(rede_neural.module_, input_size=(1, 30))


Layer (type:depth-idx)                   Output Shape              Param #
RedeNeuralModule                         [1]                       --
├─Linear: 1-1                            [1, 16]                   496
├─ReLU: 1-2                              [1, 16]                   --
├─Linear: 1-3                            [1, 16]                   272
├─ReLU: 1-4                              [1, 16]                   --
├─Linear: 1-5                            [1, 1]                    17
Total params: 785
Trainable params: 785
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00

**11) Abaixo são adicionados os parâmetros do otimizador: learning_rate e clipvalue. O que eles fazem?**

- **learning_rate (0.001)**: controla o tamanho do passo dado na descida do gradiente. Valores muito altos fazem o modelo divergir; valores muito baixos tornam o treinamento lento. 0.001 é o valor padrão do Adam.
- **clipvalue (0.5)**: limita o valor máximo dos gradientes durante o backpropagation. Evita o problema de "gradiente explosivo", onde gradientes muito grandes desestabilizam o treinamento.

In [20]:
from skorch.callbacks import Callback

class ClipGradByValue(Callback):
    def on_grad_computed(self, net, named_parameters, **kwargs):
        torch.nn.utils.clip_grad_value_(net.module_.parameters(), clip_value=0.5)

rede_neural.set_params(
    optimizer=torch.optim.Adam,
    optimizer__lr=0.001,
    criterion=nn.BCEWithLogitsLoss,
    callbacks=[ClipGradByValue()],
)


,module,<class '__mai...NeuralModule'>
,criterion,<class 'torch...thLogitsLoss'>
,train_split,<skorch.datas...001EBFFD60050>
,threshold,0.5
,optimizer,<class 'torch...im.adam.Adam'>
,lr,0.01
,max_epochs,100
,batch_size,10
,iterator_train,<class 'torch...r.DataLoader'>
,iterator_valid,<class 'torch...r.DataLoader'>
,dataset,<class 'skorc...aset.Dataset'>


**12) Teste novamente a RNA. Aumentar a quantidade de camadas melhorou ou piorou os resultados? Explique o que aconteceu com a RNA e porque.**

Com mais uma camada oculta, a rede ganha mais capacidade de aprendizado, podendo capturar padrões mais complexos. Em geral, o desempenho no treino melhora, porém pode ocorrer **overfitting**: o modelo aprende os dados de treino com excessiva precisão mas generaliza pior para dados novos. O resultado no conjunto de teste pode melhorar levemente ou se manter estável; se piorar, é sinal de overfitting.

In [21]:
rede_neural.fit(X_tr, y_tr)


,module,<class '__mai...NeuralModule'>
,criterion,<class 'torch...thLogitsLoss'>
,train_split,<skorch.datas...001EBFFD60050>
,threshold,0.5
,optimizer,<class 'torch...im.adam.Adam'>
,lr,0.01
,max_epochs,100
,batch_size,10
,iterator_train,<class 'torch...r.DataLoader'>
,iterator_valid,<class 'torch...r.DataLoader'>
,dataset,<class 'skorc...aset.Dataset'>


In [22]:
previsoes = rede_neural.predict_proba(X_te)[:, 1]


In [23]:
acuracia = rede_neural.score(X_te, y_te)
print(f'Acurácia binária no teste: {acuracia:.4f}')


Acurácia binária no teste: 0.7133


## K-Fold Cross Validation

**13) Explique como esta rede foi configurada e como é possível chegar no resultado dela. O que é necessário fazer?**

A validação cruzada K-Fold divide o conjunto de dados em K partes (folds) iguais. Em cada iteração, K-1 folds são usados para treino e 1 fold para validação, rotacionando-se qual fold é o de validação. Ao final, obtêm-se K métricas de acurácia, cujas média e desvio padrão representam o desempenho esperado e a variabilidade do modelo. Aqui utilizamos K=10 folds, o que significa 10 treinos completos da rede. A RNA é encapsulada com `KerasClassifier` para funcionar com a API do scikit-learn.

**14) Calcule também o Desvio Padrão dos resultados para avaliar o modelo. O que é possível concluir com esse resultado?**

Um desvio padrão baixo indica que o modelo tem desempenho consistente entre os folds, ou seja, generaliza bem. Um desvio padrão alto sugere que o modelo é instável e sensível a quais dados são usados para treino, podendo indicar overfitting ou poucos dados.

In [24]:
from skorch import NeuralNetBinaryClassifier
from skorch.callbacks import Callback
from sklearn.model_selection import cross_val_score
import torch.nn as nn
import torch


In [25]:
class ClipGradByValue(Callback):
    def on_grad_computed(self, net, named_parameters, **kwargs):
        torch.nn.utils.clip_grad_value_(net.module_.parameters(), clip_value=0.5)

def criar_rede():
    return NeuralNetBinaryClassifier(
        module=RedeNeuralModule,
        max_epochs=100,
        batch_size=10,
        optimizer=torch.optim.Adam,
        optimizer__lr=0.001,
        criterion=nn.BCEWithLogitsLoss,
        callbacks=[ClipGradByValue()],
        verbose=0,
    )


In [26]:
rede_neural = criar_rede()


In [27]:
X_np = X.values.astype(np.float32)
y_np = y.values.astype(np.float32).ravel()
resultados = cross_val_score(estimator=rede_neural, X=X_np, y=y_np, cv=10, scoring='accuracy')


In [28]:
resultados

array([0.77192982, 0.85964912, 0.64912281, 0.68421053, 0.66666667,
       0.80701754, 0.64912281, 0.45614035, 0.78947368, 0.75      ])

In [29]:
print(f'Média de acurácia: {resultados.mean():.4f}')
print(f'Desvio padrão:     {resultados.std():.4f}')

Média de acurácia: 0.7083
Desvio padrão:     0.1086


## Overfitting e Dropout

**14) Como vimos, o modelo anterior sofreu overfitting. Para reduzir esse problema, aplicamos a técnica de regularização conhecida como dropout. Aplique o dropout de 20% na primeira e segunda camada oculta. O que acontece com os resultados? E o Desvio Padrão?**

O Dropout desativa aleatoriamente 20% dos neurônios a cada passo de treinamento. Isso impede que a rede dependa demais de neurônios específicos, forçando o aprendizado de representações mais robustas e distribuídas. Espera-se que a acurácia média se mantenha boa e que o desvio padrão reduza, indicando menor variabilidade e melhor generalização.

In [30]:
class RedeNeuralDropoutModule(nn.Module):
    def __init__(self, n_features=30, neurons=16, dropout=0.2):
        super().__init__()
        self.oculta1 = nn.Linear(n_features, neurons)
        self.drop1 = nn.Dropout(dropout)
        self.oculta2 = nn.Linear(neurons, neurons)
        self.drop2 = nn.Dropout(dropout)
        self.relu = nn.ReLU()
        self.saida = nn.Linear(neurons, 1)
        for camada in (self.oculta1, self.oculta2, self.saida):
            nn.init.uniform_(camada.weight)

    def forward(self, X):
        x = self.drop1(self.relu(self.oculta1(X)))
        x = self.drop2(self.relu(self.oculta2(x)))
        x = self.saida(x)
        return x.squeeze(-1)

def criar_rede_dropout():
    return NeuralNetBinaryClassifier(
        module=RedeNeuralDropoutModule,
        max_epochs=100,
        batch_size=10,
        optimizer=torch.optim.Adam,
        optimizer__lr=0.001,
        criterion=nn.BCEWithLogitsLoss,
        callbacks=[ClipGradByValue()],
        verbose=0,
    )


In [31]:
rede_neural = criar_rede_dropout()


In [32]:
resultados = cross_val_score(estimator=rede_neural, X=X_np, y=y_np, cv=10, scoring='accuracy')


In [33]:
resultados

array([0.57894737, 0.61403509, 0.71929825, 0.71929825, 0.70175439,
       0.8245614 , 0.71929825, 0.84210526, 0.77192982, 0.67857143])

In [34]:
print(f'Média de acurácia (com Dropout): {resultados.mean():.4f}')
print(f'Desvio padrão (com Dropout):     {resultados.std():.4f}')

Média de acurácia (com Dropout): 0.7170
Desvio padrão (com Dropout):     0.0785


## Tuning dos Hiperparâmetros (técnicas de ajuste)

Hyperparameter Tuning (Ajuste de Hiperparâmetros) é o processo geral de encontrar os melhores valores para os hiperparâmetros de um modelo de machine learning.
Exemplos de hiperparâmetros:

- Taxa de aprendizado (learning rate)
- Número de árvores em um RandomForest
- Número de camadas ou neurônios em uma rede neural
- Tipo de kernel em um SVM

O tuning pode ser feito por diferentes técnicas, como:

- Grid Search
- Random Search
- Bayesian Optimization, etc.

**15) Descreva como a RNA foi configurada para fazer o processo de tuning.**

A função `criar_rede` foi parametrizada para aceitar os hiperparâmetros como argumentos (`optimizer`, `loss`, `kernel_initializer`, `activation`, `neurons`). O `KerasClassifier` encapsula essa função e o `GridSearchCV` testa todas as combinações definidas em `parametros`, avaliando cada configuração com validação cruzada de 5 folds. Ao final, `best_params_` e `best_score_` retornam a combinação de hiperparâmetros que maximizou a acurácia média.

In [35]:
class RedeNeuralTunavelModule(nn.Module):
    def __init__(self, n_features=30, neurons=16, activation=nn.ReLU):
        super().__init__()
        self.oculta1 = nn.Linear(n_features, neurons)
        self.drop1 = nn.Dropout(0.2)
        self.oculta2 = nn.Linear(neurons, neurons)
        self.drop2 = nn.Dropout(0.2)
        self.ativacao = activation()
        self.saida = nn.Linear(neurons, 1)

    def forward(self, X):
        x = self.drop1(self.ativacao(self.oculta1(X)))
        x = self.drop2(self.ativacao(self.oculta2(x)))
        x = self.saida(x)
        return x.squeeze(-1)

rede_neural = NeuralNetBinaryClassifier(
    module=RedeNeuralTunavelModule,
    criterion=nn.BCEWithLogitsLoss,
    verbose=0,
)


In [36]:
# rede_neural já criado na célula acima


In [37]:
parametros = {
    'batch_size': [10, 30],
    'max_epochs': [50, 100],
    'optimizer': [torch.optim.Adam, torch.optim.SGD],
    'module__activation': [nn.ReLU, nn.Tanh],
    'module__neurons': [16, 8],
}


In [38]:
# caso tiver problema com a configuração anterior, testar essa (espaço de busca reduzido)
parametros = {
    'batch_size': [10, 30],
    'max_epochs': [50],
    'optimizer': [torch.optim.Adam],
    'module__activation': [nn.ReLU],
    'module__neurons': [16],
}


In [39]:
parametros

{'batch_size': [10, 30],
 'max_epochs': [50],
 'optimizer': [torch.optim.adam.Adam],
 'module__activation': [torch.nn.modules.activation.ReLU],
 'module__neurons': [16]}

In [40]:
grid_search = GridSearchCV(estimator=rede_neural, param_grid=parametros,
                           scoring='accuracy', cv=5)


In [41]:
grid_search = grid_search.fit(X_np, y_np)


In [42]:
print(grid_search)

GridSearchCV(cv=5,
             estimator=NeuralNetBinaryClassifier(_params_to_validate=set(), batch_size=128, callbacks=None, compile=False, dataset=<class 'skorch.dataset.Dataset'>, device='cpu', iterator_train=<class 'torch.utils.data.dataloader.DataLoader'>, iterator_valid=<class 'torch.utils.data.dataloader.DataLoader'>, lr=0.01, max_epochs=10, module=<class '__main__.Re...Module'>, optimizer=<class 'torch.optim.sgd.SGD'>, predict_nonlinearity='auto', torch_load_kwargs=None, use_caching='auto', verbose=0, warm_start=False),
             param_grid={'batch_size': [10, 30], 'max_epochs': [50],
                         'module__activation': [<class 'torch.nn.modules.activation.ReLU'>],
                         'module__neurons': [16],
                         'optimizer': [<class 'torch.optim.adam.Adam'>]},
             scoring='accuracy')


In [43]:
melhores_parametros = grid_search.best_params_
print(melhores_parametros)

{'batch_size': 30, 'max_epochs': 50, 'module__activation': <class 'torch.nn.modules.activation.ReLU'>, 'module__neurons': 16, 'optimizer': <class 'torch.optim.adam.Adam'>}


In [44]:
melhor_precisao = grid_search.best_score_
print(melhor_precisao)

0.7607980127309424


**16) É possível melhorar ainda mais a acurácia da RNA? Como?**

Sim. Algumas estratégias possíveis:
- **Aumentar o espaço de busca do Grid Search**: testar mais combinações de neurônios, camadas, otimizadores e taxas de aprendizado.
- **Normalização dos dados**: aplicar StandardScaler ou MinMaxScaler nas features pode acelerar a convergência e melhorar a acurácia.
- **Aumentar épocas**: mais épocas permitem que a rede aprenda melhor, desde que combinado com dropout para evitar overfitting.
- **Adicionar mais camadas**: arquiteturas mais profundas capturam padrões mais complexos.
- **Batch Normalization**: normaliza as ativações entre camadas, estabilizando e acelerando o treinamento.
- **Early Stopping**: parar o treinamento quando a loss de validação para de melhorar, evitando overfitting.

# Para salvar o seu modelo de RNA

In [47]:
# Treina o modelo final com os melhores parâmetros antes de salvar
classificador = NeuralNetBinaryClassifier(
    module=RedeNeuralDropoutModule,
    max_epochs=100,
    batch_size=10,
    optimizer=torch.optim.Adam,
    optimizer__lr=0.001,
    criterion=nn.BCEWithLogitsLoss,
    callbacks=[ClipGradByValue()],
    verbose=0,
)
classificador.fit(X_tr, y_tr)


,module,<class '__mai...ropoutModule'>
,criterion,<class 'torch...thLogitsLoss'>
,train_split,<skorch.datas...001EBFFD60050>
,threshold,0.5
,optimizer,<class 'torch...im.adam.Adam'>
,lr,0.01
,max_epochs,100
,batch_size,10
,iterator_train,<class 'torch...r.DataLoader'>
,iterator_valid,<class 'torch...r.DataLoader'>
,dataset,<class 'skorc...aset.Dataset'>


In [48]:
import os

# Salva na pasta do usuário (evita problemas de permissão em pastas do sistema/OneDrive)
caminho_modelo = os.path.join(os.path.expanduser('~'), 'classificador_breast.pt')
print('Salvando em:', caminho_modelo)
classificador.save_params(f_params=caminho_modelo)


Salvando em: C:\Users\tiago\classificador_breast.pt


In [49]:
classificador_novo = NeuralNetBinaryClassifier(
    module=RedeNeuralDropoutModule,
    criterion=nn.BCEWithLogitsLoss,
)
classificador_novo.initialize()
classificador_novo.load_params(f_params=caminho_modelo)


In [50]:
torch_summary(classificador_novo.module_, input_size=(1, 30))


Layer (type:depth-idx)                   Output Shape              Param #
RedeNeuralDropoutModule                  [1]                       --
├─Linear: 1-1                            [1, 16]                   496
├─ReLU: 1-2                              [1, 16]                   --
├─Dropout: 1-3                           [1, 16]                   --
├─Linear: 1-4                            [1, 16]                   272
├─ReLU: 1-5                              [1, 16]                   --
├─Dropout: 1-6                           [1, 16]                   --
├─Linear: 1-7                            [1, 1]                    17
Total params: 785
Trainable params: 785
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00